In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed
import glob, os
import glob




In [2]:
def calculate_price_spread(df):
    """
    For test data: calculate price spread for every row.
    Computes price_spread as the difference between the maximum and minimum of all price columns,
    then adds spread values as 'price_spread' column to the dataframe.
    """    
    # calculate price spread for all rows
    df=df.copy()
    df['price_spread'] = df[['bid_price1', 'bid_price2', 'ask_price1', 'ask_price2']].max(axis=1) - df[['bid_price1', 'bid_price2', 'ask_price1', 'ask_price2']].min(axis=1)
    return df

def calculate_volume_impbalance(df):
    """
    For test data: calculate volume imbalance for every row.
    Computes volume imbalance as the difference between total bid size and total ask size,
    then adds imbalance values as 'volume_imbalance' column to the dataframe.
    """    
    
    # calculate volume imbalance for all rows
    df=df.copy()
    df['volume_imbalance'] = df['bid_size1'] + df['bid_size2'] - df['ask_size1'] - df['ask_size2']  
    return df

def calculate_bid_ask_spread(df):
    """
    For test data: calculate bid-ask spread for every row.
    Computes best_bid_price and best_ask_price from raw price columns,
    then adds spread values as 'bid_ask_spread' column to the dataframe.
    """
    df=df.copy()
    df['bid_ask_spread'] = (df['ask_price1']/df['bid_price1']) - 1
    
    return df

def calculate_volume(df):
    """
    For test data: calculate total volume for every row.
    Computes best_bid_size and best_ask_size from raw size columns,
    then adds total volume values as 'total_volume' column to the dataframe.
    """
    df=df.copy()
    df['total_volume'] = df[['bid_size1', 'bid_size2', 'ask_size1', 'ask_size2']].sum(axis=1)
    
    # drop intermediate columns if not needed later
    return df


def add_wap_at_zero(df):
    df=df.copy()
    df['wap'] = (df['bid_price1'] * df['ask_size1'] + 
                        df['ask_price1'] * df['bid_size1']) / (
                        df['bid_size1'] + df['ask_size1'])
    return df



In [4]:
out_dir = "individual_book_train_denorm"
os.makedirs(out_dir, exist_ok=True)


def fill_missing_seconds_in_bucket(df):
    """Ensure every time_id has rows for all 0..599 seconds_in_bucket, forward-filling values."""
    # Keep original order semantics per time_id and fill gaps with prior available row.
    df = df.sort_values(['time_id', 'seconds_in_bucket'])
    filled = (
        df.groupby('time_id', group_keys=False)
          .apply(lambda g: g.set_index('seconds_in_bucket')
                         .reindex(range(600))
                         .ffill()
                         .assign(time_id=g['time_id'].iloc[0]))
          .reset_index()
    )
    return filled


def process_file(path, out_dir):
    fname = os.path.basename(path)
    out_path = os.path.join(out_dir, fname)
    try:
        
        df = pd.read_csv(path)
        df = add_wap_at_zero(df)
        df = calculate_bid_ask_spread(df)
        df = calculate_volume(df)
        df = calculate_price_spread(df)
        df = calculate_volume_impbalance(df)
        df = df[['time_id', 'seconds_in_bucket', 'wap', 'bid_ask_spread', 'total_volume', 'price_spread', 'volume_imbalance']]
        df = fill_missing_seconds_in_bucket(df)
        df.to_csv(out_path, index=False)
        return (fname, 'ok')
    except Exception as e:
        return (fname, f'error: {e}')

files = glob.glob(r"individual_book_train\stock_*.csv")
print(f"Found {len(files)} files to process")

# Choose number of workers: -1 uses all CPUs. If I/O bound, reduce this.
n_jobs = -1
results = Parallel(n_jobs=n_jobs, verbose=5)(delayed(process_file)(p, out_dir) for p in files)

# Summary
oks = [r for r in results if r[1] == 'ok']
errs = [r for r in results if r[1] != 'ok']
print(f"Completed: {len(oks)}; Failed: {len(errs)}")
if errs:
    print('Failures (first 10):')
    for fn, msg in errs[:10]:
        print('-', fn, msg)

Found 112 files to process


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done  32 tasks      | elapsed:  2.2min
[Parallel(n_jobs=-1)]: Done  96 out of 112 | elapsed:  7.4min remaining:  1.2min


Completed: 112; Failed: 0


[Parallel(n_jobs=-1)]: Done 112 out of 112 | elapsed:  8.3min finished
